In [ ]:
# Cell: g4 Paper — Leaderboard Noise Floor & Minimum Detectable Effect (MDE)
# CPU-only, zero network. Reads embedded ledger.csv data + anonymized leaderboard
# counts (score -> number of teams; no names). Regenerates the noise figure bands
# and MDE numbers cited in the paper (§5.1, Table 1 notes, fig4).
import math
import json as _json

try:
    import numpy as np
    HAVE_NUMPY = True
except Exception:  # pragma: no cover - numpy is present on the Kaggle CPU image
    HAVE_NUMPY = False

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False

# ---------------------------------------------------------------------------
# Embedded ledger.csv (research/paper_figs/ledger.csv) — scored rows only.
# Columns: ref, score, tasks_passed (score*58 rounded), scale.
# exp_think1024 is pending scoring (score None) and excluded from noise math.
# ---------------------------------------------------------------------------
LEDGER = [
    ("exp01",        0.05,  3, "31B"),
    ("exp02",        0.08,  5, "31B"),
    ("exp03",        0.17, 10, "31B"),
    ("exp05",        0.20, 12, "31B"),
    ("exp08",        0.17, 10, "31B"),
    ("exp10",        0.17, 10, "31B"),
    ("57697341",     0.13,  8, "31B"),
    ("exp12",        0.17, 10, "31B"),
    ("57836949",     0.17, 10, "31B"),   # E0 control: byte-identical exp05 rebuild
    ("exp_sampling", 0.13,  8, "31B"),
    ("exp_think1024",None, None, "31B"),  # pending scoring
    ("exp11_e4b",    0.06,  4, "E4B"),
    ("e4b-v2",       0.05,  3, "E4B"),
    ("e4b-v4",       0.05,  3, "E4B"),
    ("exp14",        0.15,  9, "31B"),
    ("exp15",        0.13,  8, "31B"),
]
N_TASKS = 58  # public leaderboard tasks per run

# ---------------------------------------------------------------------------
# Anonymized leaderboard counts: (public score, number of teams). No names.
# From fig4_data.csv / paper draft §5 top-20 cluster.
# ---------------------------------------------------------------------------
LEADERBOARD_COUNTS = [
    (0.22, 8),
    (0.24, 11),
    (0.29, 1),
]

Z_ALPHA_2 = 1.96   # two-sided alpha = 0.05
Z_BETA_80 = 0.842  # 80% power


def binom_se(p, n=N_TASKS):
    """Binomial standard error for a resolution rate p over n tasks."""
    if p is None:
        return None
    return math.sqrt(p * (1.0 - p) / n)


def tasks_to_score(t):
    return t / N_TASKS


print("=" * 70)
print("g4 Paper — Leaderboard Noise Floor & MDE (CPU, zero network)")
print("=" * 70)
print(f"Public leaderboard tasks per run: {N_TASKS}")
print(f"1 resolved task = 1/{N_TASKS} = {tasks_to_score(1):.5f} score points")
print()

# --------------------------------------------------------------- noise floor
scored = [(ref, s, t, sc) for ref, s, t, sc in LEDGER if s is not None]
print(f"Scored ledger rows: {len(scored)} (exp_think1024 pending, excluded)")

# per-run binomial SEs
print("\n--- Per-run binomial SE = sqrt(p(1-p)/58) ---")
for ref, s, t, sc in scored:
    print(f"  {ref:14s} scale={sc:3s} score={s:.2f} tasks={t:2d}  SE={binom_se(s):.4f} "
          f"(±1.96 SE = ±{1.96*binom_se(s):.4f} = ±{1.96*binom_se(s)*N_TASKS:.1f} tasks)")

# pooled converged 31B rate (exp05-family: exp05, exp03, exp08, exp10, exp12, E0, exp14, exp_sampling)
conv31b = [t for ref, s, t, sc in scored if sc == "31B" and ref in
           ("exp05", "exp03", "exp08", "exp10", "exp12", "57836949", "exp14", "exp_sampling")]
pooled_tasks = sum(conv31b)
pooled_denom = len(conv31b) * N_TASKS
pooled_rate = pooled_tasks / pooled_denom
pooled_se = math.sqrt(pooled_rate * (1 - pooled_rate) / pooled_denom)
print("\n--- Pooled converged 31B rate (8 runs) ---")
print(f"  tasks = {conv31b}  sum = {pooled_tasks}")
print(f"  pooled rate = {pooled_tasks}/{pooled_denom} = {pooled_rate:.4f} "
      f"({pooled_rate*100:.1f}%)")
print(f"  pooled SE = {pooled_se:.4f}  (±1.96 = ±{1.96*pooled_se:.4f} = "
      f"±{1.96*pooled_se*N_TASKS:.1f} tasks)")

# cross-split SE (single run, n=58) at p=0.179 (paper §5.1)
cross_p = 0.179
cross_se = math.sqrt(cross_p * (1 - cross_p) / N_TASKS)
print("\n--- Cross-split public-to-private SE (n=58, p=0.179) ---")
print(f"  SE = {cross_se:.4f}  (~{cross_se*N_TASKS:.1f} tasks)")
print(f"  ±1.96 SE band = ±{1.96*cross_se:.4f} = ±{1.96*cross_se*N_TASKS:.1f} tasks")

# within-split SD (exp05 family: [12,10,10,10,10])
exp05_fam = [12, 10, 10, 10, 10]
if HAVE_NUMPY:
    within_sd = float(np.std(exp05_fam, ddof=1))
else:
    m = sum(exp05_fam) / len(exp05_fam)
    within_sd = math.sqrt(sum((x - m) ** 2 for x in exp05_fam) / (len(exp05_fam) - 1))
print("\n--- Within-split run-to-run SD (exp05 family tasks [12,10,10,10,10]) ---")
print(f"  SD = {within_sd:.2f} tasks (~{within_sd/N_TASKS:.4f} score)")

# --------------------------------------------------------------- MDE numbers
print("\n" + "=" * 70)
print("Minimum Detectable Effect (MDE) at 80% power, alpha=0.05 (two-sided)")
print("Same-split two-sample: MDE = 2.802 * sqrt(2) * SD = 2.802 * 1.259 = 3.53 tasks")
print("Cross-split binomial:  MDE = 2.802 * SE = 2.802 * 0.0503 * 58 = 8.18 tasks")
print("=" * 70)
# Same-split MDE: two-sample comparison against anchor (empirical SD = 0.894 tasks)
se_same_diff = math.sqrt(2) * within_sd  # 1.265 tasks
mde_same_tasks = (Z_ALPHA_2 + Z_BETA_80) * se_same_diff  # 3.54 tasks
mde_same_score = mde_same_tasks / N_TASKS

# Cross-split MDE: single-run binomial SE = 0.0503 (n=58, p=0.179)
mde_cross_score = (Z_ALPHA_2 + Z_BETA_80) * cross_se
mde_cross_tasks = mde_cross_score * N_TASKS  # 8.18 tasks

print(f"  Same-split MDE (SD = {within_sd:.2f} tasks, 5 runs):  3.5 tasks ({mde_same_score:.4f} score)")
print(f"  Cross-split MDE (SE = {cross_se:.4f}, n = {N_TASKS}):   8.2 tasks ({mde_cross_score:.4f} score)")
print("  Note: Pooled-SE MDE (2.8 tasks) removed (violates task independence assumption).")
print("  => Any leaderboard delta of <= ~2 tasks is within run-to-run noise;")
print("     cross-split claims require > 8.2 tasks (> 0.141 score) to resolve with 80% power.")

# --------------------------------------------------- anonymized leaderboard
print("\n" + "=" * 70)
print("Anonymized public leaderboard (top-20 cluster): score -> # teams (no names)")
print("=" * 70)
for score, nteams in LEADERBOARD_COUNTS:
    tasks = score * N_TASKS
    print(f"  score={score:.2f}  teams={nteams}  (~{tasks:.1f} tasks)")
top_tasks = [s * N_TASKS for s, _ in LEADERBOARD_COUNTS]
print(f"  Top-20 task span: {min(top_tasks):.0f} to {max(top_tasks):.0f} tasks "
      f"(= {max(top_tasks)-min(top_tasks):.0f}-task band)")
print(f"  Delta ≤ 2 tasks (≤0.034) is within cross-split noise "
      f"(±1.96 SE = ±{1.96*cross_se*N_TASKS:.1f} tasks).")

# E0 control highlight
print("\n" + "=" * 70)
print("E0 reproduction control: byte-identical exp05 (sha 2ba7f9b1…)")
print("=" * 70)
print("  exp05 scored 0.20 (12 tasks); E0 byte-identical rebuild scored 0.17 (10 tasks).")
print("  Same artifact, different score -> 0.20 was an upper-tail binomial draw,")
print("  not a permanent gain. Pooled 31B ≈ 0.170 is the plateau.")

# --------------------------------------------------------------- noise figure
if HAVE_MPL:
    import os
    os.makedirs("/kaggle/working", exist_ok=True)
    xs = [s for _, s, _, _ in scored]
    ses = [binom_se(s) for _, s, _, _ in scored]
    fig, ax = plt.subplots(figsize=(9, 4.2), dpi=150)
    ax.errorbar(xs, [s for _, s, _, _ in scored], yerr=ses,
                fmt="o", color="#1f77b4", ecolor="#94a3b8", capsize=4, label="scored runs")
    ax.axhline(pooled_rate, color="#1f77b4", ls="--", lw=1.4,
               label=f"pooled 31B = {pooled_rate:.3f}")
    ax.axhspan(pooled_rate - 1.96 * cross_se, pooled_rate + 1.96 * cross_se,
               color="#f1f5f9", alpha=0.9,
               label=f"±1.96 cross-split SE (±{1.96*cross_se*N_TASKS:.0f} tasks)")
    ax.set_xlabel("Public leaderboard score (Pass@1)")
    ax.set_ylabel("Score")
    ax.set_title("Leaderboard Noise Floor (binomial SE bands, n=58 tasks/run)")
    ax.set_ylim(0, max(xs + [0.25]) * 1.15)
    ax.grid(axis="y", alpha=0.4)
    ax.legend(fontsize=8)
    fig.tight_layout()
    out = "/kaggle/working/noise_figure.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"\nNoise figure saved to {out}")
else:
    print("\nmatplotlib unavailable; skipping noise figure PNG.")

print("\n[DONE] noise + MDE regeneration complete.")